In [1]:
# define categories for classifier

relevant_categories = [
    "Cap On Liability", "Termination For Convenience", "Governing Law",
    "Anti-Assignment", "Change Of Control", "Renewal Term",
    "Exclusivity", "Non-Compete", "Notice Period To Terminate Renewal", "Liquidated Damages"
]

relevant_categories = [category.casefold() for category in relevant_categories]

In [2]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent  # if the notebook's working directory is notebooks/
sys.path.insert(0, str(repo_root))

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MultiLabelBinarizer
from src.chunking import get_tokenizer, json_to_df_new
from src.baseline import fit_baseline_model
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import numpy as np

c:\Users\fulle\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
tokenizer = get_tokenizer()

def prepare_data(path, tokenizer, mlb, relevant_categories):
    chunks = json_to_df_new(path, tokenizer)
    chunks['categories'] = chunks['categories'].apply(
        lambda labels: [label for label in labels if label in relevant_categories] or ["None"]
    )
    X = chunks['text'].fillna('').astype(str)
    y = mlb.fit_transform(chunks['categories'])
    return X, y


In [5]:
project_root = Path.cwd().parent

In [6]:
model_dir = project_root / "contract-category-model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 548.01it/s]


In [7]:
mlb = MultiLabelBinarizer(classes=relevant_categories + ["None"])
X_test, y_test = prepare_data(project_root / 'data/test.json', tokenizer, 
                              mlb, relevant_categories) 

# #bootstrap for 95% confidence threshold
# scores_baseline = []
# scores_model = []
# for _ in range(500):
#     random_indices = np.random.choice(range(len(X_test)),size=len(X_test),replace=True)
#     X_test_new = X_test[random_indices]
#     y_test_new = y_test[random_indices]

#     X_test_new_vec = vectorizer.fit_transform(X_test_new)
#     scores_baseline.append(accuracy_score(y_test_new,svc_classifier.predict(X_test_new_vec)))
#     scores_model.append(accuracy_score(y_test_new, model.predict(X_test_new)))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2413 > 512). Running this sequence through the model will result in indexing errors


In [8]:
svc_classifier, vectorizer = fit_baseline_model()

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (11248 > 512). Running this sequence through the model will result in indexing errors


In [9]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# Predict once on the same, untouched test set.
# The transformer call assumes your model accepts raw texts; use its normal
# inference/preprocessing path if it expects tokenized inputs instead.
y = np.asarray(y_test)
baseline_pred = np.asarray(
    svc_classifier.predict(vectorizer.transform(X_test))
)
transformer_pred = np.asarray(model.predict(X_test))

rng = np.random.default_rng(42)
n = len(y)
n_bootstrap = 5000

accuracy_deltas = []
micro_f1_deltas = []

for _ in range(n_bootstrap):
    idx = rng.integers(0, n, size=n)  # same resampled rows for both models
    y_sample = y[idx]

    accuracy_deltas.append(
        accuracy_score(y_sample, transformer_pred[idx])
        - accuracy_score(y_sample, baseline_pred[idx])
    )
    micro_f1_deltas.append(
        f1_score(y_sample, transformer_pred[idx], average="micro", zero_division=0)
        - f1_score(y_sample, baseline_pred[idx], average="micro", zero_division=0)
    )

for name, metric, delta in [
    ("Exact-match accuracy", accuracy_score, accuracy_deltas),
    ("Micro F1", lambda a, b: f1_score(a, b, average="micro", zero_division=0),
     micro_f1_deltas),
]:
    baseline_score = metric(y, baseline_pred)
    transformer_score = metric(y, transformer_pred)
    low, high = np.percentile(delta, [2.5, 97.5])

    print(
        f"{name}: baseline={baseline_score:.3f}, "
        f"transformer={transformer_score:.3f}, "
        f"transformer - baseline={transformer_score - baseline_score:+.3f}, "
        f"95% bootstrap CI for difference=[{low:+.3f}, {high:+.3f}]"
    )

AttributeError: 'BertForSequenceClassification' object has no attribute 'predict'